In [1]:
import numpy as np
import pandas as pd

def train_val_split(X, y, val_ratio=0.2, seed=42):
    np.random.seed(seed)
    idx = np.random.permutation(len(X))
    split = int(len(X) * (1 - val_ratio))

    train_idx = idx[:split]
    val_idx = idx[split:]

    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

def save_submission(predictions, filename="Polynomial_Pred.csv", column_name="target"):
    predictions = predictions.reshape(-1)
    df = pd.DataFrame({column_name: predictions})
    df.to_csv(filename, index=False)
    print(f"Submission file saved as '{filename}'")

def load_train_data(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

def load_test_data(path):
    df = pd.read_csv(path)
    return df.values

def normalize_train(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma 
    return X_norm, mu, sigma

def normalize_test(X, mu, sigma):
    return (X - mu) / sigma

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, w, b):
    m = X.shape[0]
    z = X @ w + b
    h = sigmoid(z)
    cost = -np.mean( y * np.log(h + 1e-15) + (1 - y) * np.log(1 - h + 1e-15))
    return cost

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    z = X @ w + b
    h = sigmoid(z)
    errors = h - y

    dj_dw = (X.T @ errors) / m
    dj_db = np.sum(errors) / m

    return dj_db, dj_dw

def logistic_regression(X, y, learning_rate=0.01, num_iters=1000):
    X, mu, sigma = normalize_train(X)

    n = X.shape[1]
    w = np.zeros(n)
    b = 0.0
    J_history = []

    for i in range(num_iters):
        dj_db, dj_dw = compute_gradient(X, y, w, b)

        w -= learning_rate * dj_dw
        b -= learning_rate * dj_db

        if i % 50 == 0:
            cost = compute_cost(X, y, w, b)
            J_history.append(cost)
            print(f"Iteration {i:4d}: Cost {cost:.4f}")

    return w, b, mu, sigma, J_history

def macro_f1_score(y_true, y_pred):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)

    classes = np.unique(y_true)
    f1_scores = []

    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))

        precision = tp / (tp + fp + 1e-15)
        recall = tp / (tp + fn + 1e-15)

        f1 = 2 * precision * recall / (precision + recall + 1e-15)
        f1_scores.append(f1)

    return np.mean(f1_scores)

def predict_probability(X, w, b, mu, sigma):
    X = normalize_test(X, mu, sigma)
    return sigmoid(X @ w + b)

def predict(X, w, b, mu, sigma, threshold=0.5):
    probs = predict_probability(X, w, b, mu, sigma)
    return (probs >= threshold).astype(int)

#To run the model
X, y= load_train_data("train_binary.csv")
X_test = load_test_data("test_binary.csv")
X_train, X_val, y_train, y_val = train_val_split(X,y)
w, b, mu, sigma, _ = logistic_regression(X_train, y_train, learning_rate=0.01, num_iters=1000)
y_pred = predict(X_test, w, b, mu, sigma)
print(y_pred[:5])

#To calculate Macro F1 score
y_val_pred = predict(X_val, w, b, mu, sigma)
macro_f1_val = macro_f1_score(y_val, y_val_pred)
print(f"Validation Macro F1 Score: {macro_f1_val:.4f}")

# Prediction file
save_submission(
    predictions=y_pred,
    filename="Logistic_pred.csv",
    column_name="target"  
)

Iteration    0: Cost 0.6857
Iteration   50: Cost 0.4429
Iteration  100: Cost 0.3300
Iteration  150: Cost 0.2658
Iteration  200: Cost 0.2240
Iteration  250: Cost 0.1946
Iteration  300: Cost 0.1726
Iteration  350: Cost 0.1555
Iteration  400: Cost 0.1418
Iteration  450: Cost 0.1306
Iteration  500: Cost 0.1212
Iteration  550: Cost 0.1132
Iteration  600: Cost 0.1063
Iteration  650: Cost 0.1003
Iteration  700: Cost 0.0950
Iteration  750: Cost 0.0903
Iteration  800: Cost 0.0861
Iteration  850: Cost 0.0823
Iteration  900: Cost 0.0789
Iteration  950: Cost 0.0758
[0 1 1 1 1]
Validation Macro F1 Score: 0.9959
Submission file saved as 'Logistic_pred.csv'
